# VertiRTA-X8-240 — 03. TR/EN Neuro-Symbolic Co-Pilot (`Qwen3` BF16 SFT -> Z3-GRPO & `VertiBench-TR/EN`)

**Recommended GPU:** **Google Colab `L4 (24 GB)`** for **`Qwen/Qwen3-4B`** BF16 LoRA (`r=32, alpha=64`) SFT -> Z3-GRPO (`G=8`) + `Q4_K_M` GGUF + 4-model `VertiBench-TR/EN` eval.
**Optional `A100 (40/80 GB)` Toggle:** Enable `ENABLE_A100_8B_TOGGLE = True` **ONLY** if training **`Qwen/Qwen3-8B`** in full BF16.
**Do NOT allocate `G4 (96 GB)`:** Opening a 96 GB G4 for a 4B job wastes ~4x Compute Units; the startup cell explicitly warns you if a `>= 60 GB` GPU is attached for `Qwen3-4B`.

### License & Safety Firewall (Enforced Programmatically)
- **Fine-Tuning Allowlist (Apache-2.0 ONLY):** `Qwen/Qwen3-4B` (default on L4) and `Qwen/Qwen3-8B` (optional A100).
- **Zero-Shot Eval Baselines:** `Qwen/Qwen2.5-7B-Instruct` and `google/gemma-3-4b-it` (strictly zero-shot + GBNF + Z3).
- **Hard-Banned:** `CohereForAI/aya-expanse-8b` (`CC-BY-NC-4.0`).
- **Telemetry & Anti-Contamination Lock:** `400 V / 22 kWh` HV pack, `0` seed overlap, `< 0.15` char-5-gram & token-3-gram Jaccard.

In [ ]:
import subprocess

# --- User Controls (Right-Sized Defaults) ---
ENABLE_A100_8B_TOGGLE = False  # Set True ONLY on A100 (40/80 GB) to train Qwen/Qwen3-8B
MODEL_ID = "Qwen/Qwen3-8B" if ENABLE_A100_8B_TOGGLE else "Qwen/Qwen3-4B"
USE_GSPO = False  # False => dr_grpo (0.20/0.28); True => calibrated GSPO (3e-4/4e-4)
BETA_KL = 0.04

out = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
    text=True,
).strip()
name, driver_ver, mem_mb_str = [part.strip() for part in out.splitlines()[0].split(",")]
vram_gb = float(mem_mb_str) / 1024.0
print(f"Detected GPU: {name} | Driver: {driver_ver} | VRAM: {vram_gb:.1f} GB | Target Model: {MODEL_ID}")

if MODEL_ID == "Qwen/Qwen3-4B" and vram_gb >= 60.0:
    print(
        "[COST WARNING] You accidentally opened a >=60 GB GPU (e.g., G4 96 GB / A100 80 GB) "
        "for a Qwen3-4B job! Qwen3-4B BF16 LoRA SFT -> Z3-GRPO (G=8, max_completion_length=64) "
        "runs cleanly on Colab L4 (24 GB). Disconnect and switch to L4 to save Compute Units!"
    )
elif MODEL_ID == "Qwen/Qwen3-8B":
    if vram_gb < 36.0:
        raise RuntimeError(
            f"ENABLE_A100_8B_TOGGLE=True requires A100 (40/80 GB) for full-BF16 Qwen3-8B; "
            f"detected {vram_gb:.1f} GB. Set ENABLE_A100_8B_TOGGLE=False for L4 (24 GB) Qwen3-4B."
        )
    if vram_gb >= 90.0:
        print(
            "[COST WARNING] Qwen3-8B BF16 fits comfortably in A100 (40 GB); G4 (96 GB) is unnecessary."
        )
elif vram_gb < 20.0:
    raise RuntimeError(
        f"Qwen3-4B BF16 SFT->GRPO requires >= 22 GB VRAM (Colab L4 24 GB); detected {vram_gb:.1f} GB."
    )
else:
    print("GPU tier right-sized: L4 (24 GB) for Apache-2.0 Qwen/Qwen3-4B SFT -> Z3-GRPO.")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")
except ImportError:
    print("Not running inside Google Colab; skipping drive.mount().")

REPO_URL = os.environ.get("VERTIRTA_REPO_URL", "https://github.com/vertirta/vertirta.git")
REPO_DIR = Path("/content/vertirta")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "unsloth",
        "trl",
        "datasets",
        "z3-solver",
        "vllm",
        "pyyaml",
        "numpy",
    ],
    check=True,
)

from benchmarks.vertibench_tr_en.validate_vertibench import is_model_allowed

assert is_model_allowed(MODEL_ID, "train"), f"Model {MODEL_ID} blocked by Apache-2.0 training firewall!"
assert not is_model_allowed("CohereForAI/aya-expanse-8b", "zero_shot_eval"), "CC-BY-NC guard failed!"
print(f"License gate PASSED for {MODEL_ID} (Apache-2.0).")

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

DATA_DIR = Path("/content/vertirta/benchmarks/vertibench_tr_en/data")
DRIVE_BENCH_DIR = Path("/content/drive/MyDrive/VertiRTA_Artifacts/vertibench_tr_en")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. Deterministically synthesize 1,800-row 400V/22kWh GBNF+Z3-filtered train.jsonl
subprocess.run(
    [
        sys.executable,
        "-m",
        "benchmarks.vertibench_tr_en.audit_cli",
        "build-train",
        "--out-dir",
        str(DATA_DIR),
        "--seed",
        "2026",
    ],
    check=True,
)

# 2. Copy human-authored calibration.jsonl (200) & test.jsonl (560) from Drive if present and audit
for split_file in ("calibration.jsonl", "test.jsonl"):
    src = DRIVE_BENCH_DIR / split_file
    if src.exists():
        shutil.copy2(src, DATA_DIR / split_file)

if (DATA_DIR / "calibration.jsonl").exists() and (DATA_DIR / "test.jsonl").exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "benchmarks.vertibench_tr_en.audit_cli",
            "audit",
            "--data-dir",
            str(DATA_DIR),
            "--split",
            "all",
        ],
        check=True,
    )
else:
    print(
        "[INFO] Human-authored calibration.jsonl / test.jsonl not found in Drive yet; "
        "running schema self-check only."
    )
    subprocess.run(
        [sys.executable, "-m", "benchmarks.vertibench_tr_en.validate_vertibench", "--check-schema"],
        check=True,
    )

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

DATA_DIR = Path("/content/vertirta/benchmarks/vertibench_tr_en/data")
MODEL_SLUG = "qwen3-8b" if "8B" in MODEL_ID else "qwen3-4b"
CKPT_DIR = Path(f"/content/vertirta/artifacts/v2.0.0/copilot_{MODEL_SLUG}")
DRIVE_CKPT = Path(f"/content/drive/MyDrive/VertiRTA_Artifacts/copilot_{MODEL_SLUG}")

base_cmd = [
    sys.executable,
    "-m",
    "ml.copilot.train_sft_grpo",
    "--model-id",
    MODEL_ID,
    "--train-jsonl",
    str(DATA_DIR / "train.jsonl"),
    "--output-dir",
    str(CKPT_DIR),
    "--beta",
    str(BETA_KL),
]
if USE_GSPO:
    base_cmd.append("--gspo")

# Step A: Dry-run policy & license lock verification
subprocess.run(base_cmd + ["--dry-run"], check=True)

# Step B: Full Stage-1 BF16 LoRA SFT -> Stage-2 Z3-GRPO (G=8, spawn Z3 pool)
subprocess.run(base_cmd, check=True)

DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
shutil.copytree(CKPT_DIR / "merged_bf16", DRIVE_CKPT / "merged_bf16", dirs_exist_ok=True)
print("Backed up merged BF16 checkpoint to Drive:", DRIVE_CKPT / "merged_bf16")

In [ ]:
import shutil
import subprocess
from pathlib import Path

MODEL_SLUG = "qwen3-8b" if "8B" in MODEL_ID else "qwen3-4b"
MERGED_DIR = Path(f"/content/vertirta/artifacts/v2.0.0/copilot_{MODEL_SLUG}/merged_bf16")
GGUF_OUT_DIR = Path("/content/vertirta/artifacts/v2.0.0/gguf")
LLAMA_CPP_DIR = Path("/content/llama.cpp")

if not (LLAMA_CPP_DIR / "build/bin/llama-quantize").exists():
    if not LLAMA_CPP_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP_DIR)],
            check=True,
        )
    subprocess.run(["cmake", "-S", str(LLAMA_CPP_DIR), "-B", str(LLAMA_CPP_DIR / "build"), "-DLLAMA_CURL=OFF"], check=True)
    subprocess.run(["cmake", "--build", str(LLAMA_CPP_DIR / "build"), "--target", "llama-quantize", "-j4"], check=True)

subprocess.run(
    [
        "bash",
        "ml/copilot/export_gguf.sh",
        MODEL_SLUG,
        str(MERGED_DIR),
        str(GGUF_OUT_DIR),
        str(LLAMA_CPP_DIR),
    ],
    check=True,
)

drive_gguf = Path("/content/drive/MyDrive/VertiRTA_Artifacts/gguf")
drive_gguf.mkdir(parents=True, exist_ok=True)
for fpath in GGUF_OUT_DIR.glob("*"):
    shutil.copy2(fpath, drive_gguf / fpath.name)
print("Synced Q4_K_M GGUF + SHA-256 to:", drive_gguf)

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

CAL_SCORED = Path("/content/drive/MyDrive/VertiRTA_Artifacts/vertibench_tr_en/calibration_scored.jsonl")
OUT_JSON = Path("/content/vertirta/config/conformal_calibration.json")

if CAL_SCORED.exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "ml.copilot.calibrate_conformal",
            "--calibration",
            str(CAL_SCORED),
            "--out",
            str(OUT_JSON),
            "--alpha",
            "0.05",
            "--expected-n-cal",
            "200",
        ],
        check=True,
    )
    drive_cfg = Path("/content/drive/MyDrive/VertiRTA_Artifacts/copilot_eval/conformal_calibration.json")
    drive_cfg.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(OUT_JSON, drive_cfg)
    print("Saved calibrated KnowNo q_hat JSON to:", drive_cfg)
else:
    print(
        f"[INFO] Scored calibration probabilities not yet at {CAL_SCORED}; "
        "generate GBNF log-probs on calibration.jsonl (n=200) and rerun this cell."
    )

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from benchmarks.vertibench_tr_en.eval_vertibench import EVAL_MODEL_MATRIX

TEST_JSONL = Path("/content/vertirta/benchmarks/vertibench_tr_en/data/test.jsonl")
PRED_DIR = Path("/content/drive/MyDrive/VertiRTA_Artifacts/vertibench_predictions")
EVAL_OUT_DIR = Path("/content/drive/MyDrive/VertiRTA_Artifacts/copilot_eval")
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

leaderboard = []
for model_id, role in EVAL_MODEL_MATRIX:
    slug = model_id.replace("/", "__")
    stage = "sft_z3_grpo_gbnf_z3_conformal" if role == "trained_apache2" else "zero_shot_gbnf_z3"
    pred_path = PRED_DIR / f"{slug}__{stage}.jsonl"
    out_json = EVAL_OUT_DIR / f"eval_{slug}__{stage}.json"
    if not (TEST_JSONL.exists() and pred_path.exists()):
        print(f"[SKIP] Missing test split or predictions for {model_id} ({pred_path})")
        continue
    subprocess.run(
        [
            sys.executable,
            "-m",
            "benchmarks.vertibench_tr_en.eval_vertibench",
            "--model-id",
            model_id,
            "--stage",
            stage,
            "--test-jsonl",
            str(TEST_JSONL),
            "--predictions-jsonl",
            str(pred_path),
            "--out-json",
            str(out_json),
        ],
        check=True,
    )
    leaderboard.append(json.loads(out_json.read_text(encoding="utf-8")))

(EVAL_OUT_DIR / "vertibench_4model_leaderboard.json").write_text(
    json.dumps({"models": leaderboard}, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Saved 4-model VertiBench leaderboard ({len(leaderboard)} evaluated) to {EVAL_OUT_DIR}")